# Phase 6 — Structured Streaming

**Duration:** Weeks 9–10 | **Goal:** Process data continuously as it arrives, with exactly-once guarantees.

---

### What We'll Cover

| Section | Key Skills |
| --- | --- |
| 6.1 Batch vs Streaming | Why streaming? Micro-batch architecture, `readStream` / `writeStream` |
| 6.2 Triggers | `availableNow`, `processingTime`, continuous — when does each batch fire? |
| 6.3 Output Modes | `append`, `complete`, `update` — which rows get written |
| 6.4 Checkpoints | Exactly-once guarantees, fault recovery, checkpoint directory |
| 6.5 Watermarks | Handling late data, `withWatermark()`, dropping stale records |
| 6.6 Stateful Operations | Windowed aggregations, sessionization |
| 6.7 Stream-to-Stream Joins | Joining two streams with watermarks |

### Dataset

* **Source:** Simulated event data written to `/Volumes/arnalladatabricks/pyspark_learning/raw_data/streaming_events/`
* We'll write files in batches to simulate a real stream arriving over time

---

**Pre-requisite:** Phases 2–5 completed (DataFrames, Delta Lake, Performance Tuning)

In [0]:
# ============================================================
# STEP 1: Setup — create simulated event data for streaming
# ============================================================
from pyspark.sql.functions import (
    col, current_timestamp, lit, rand, floor, expr,
    window, count, sum as spark_sum, avg, max as spark_max,
    from_json, to_json, struct, date_format
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    TimestampType, DoubleType
)
import time

# Paths
SCHEMA = "arnalladatabricks.pyspark_learning"
VOLUME_PATH = "/Volumes/arnalladatabricks/pyspark_learning/raw_data"
STREAM_SOURCE = f"{VOLUME_PATH}/streaming_events"
STREAM_CHECKPOINT = f"{VOLUME_PATH}/checkpoints/streaming_demo"
STREAM_OUTPUT = f"{VOLUME_PATH}/streaming_output"

# Clean up from previous runs
dbutils.fs.rm(STREAM_SOURCE, recurse=True)
dbutils.fs.rm(STREAM_CHECKPOINT, recurse=True)
dbutils.fs.rm(STREAM_OUTPUT, recurse=True)

# --- Define schema for our events ---
event_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("user_id", IntegerType(), False),
    StructField("event_type", StringType(), False),
    StructField("page", StringType(), False),
    StructField("event_time", TimestampType(), False),
    StructField("amount", DoubleType(), True)
])

# --- Helper function to generate a batch of events ---
def generate_events(batch_id, num_events=100):
    """Simulate a batch of user activity events."""
    df_events = spark.range(num_events).select(
        expr(f"concat('evt_', {batch_id}, '_', id)").alias("event_id"),
        (floor(rand() * 50) + 1).cast("int").alias("user_id"),
        expr("""CASE floor(rand() * 4)
            WHEN 0 THEN 'page_view'
            WHEN 1 THEN 'click'
            WHEN 2 THEN 'purchase'
            ELSE 'logout'
        END""").alias("event_type"),
        expr("""CASE floor(rand() * 5)
            WHEN 0 THEN '/home'
            WHEN 1 THEN '/products'
            WHEN 2 THEN '/cart'
            WHEN 3 THEN '/checkout'
            ELSE '/profile'
        END""").alias("page"),
        # Events within the last 5 minutes (some slightly late)
        expr(f"current_timestamp() - make_interval(0,0,0,0,0, floor(rand() * 300), 0)").alias("event_time"),
        expr("CASE WHEN rand() < 0.25 THEN round(rand() * 200, 2) ELSE null END").alias("amount")
    )
    # Write as JSON directly to source dir (streaming picks up new files)
    df_events.write.mode("append").json(STREAM_SOURCE)
    return num_events

# --- Generate first batch ---
n = generate_events(1, 200)
print(f"✅ Generated batch 1: {n} events")
print(f"   Source path: {STREAM_SOURCE}")
print(f"   Checkpoint:  {STREAM_CHECKPOINT}")

# Verify
df_check = spark.read.schema(event_schema).json(STREAM_SOURCE)
print(f"   Total events on disk: {df_check.count()}")
df_check.show(5, truncate=False)

## 6.1 — Batch vs Streaming: Same API, Different Execution

### The key insight

Structured Streaming uses the **exact same DataFrame API** you already know. The only difference:

| | Batch | Streaming |
| --- | --- | --- |
| Read | `spark.read` | `spark.readStream` |
| Write | `df.write` | `df.writeStream` |
| Execution | Runs once, processes all data | Runs continuously, processes new data as it arrives |
| Result | Static DataFrame | Unbounded table that grows over time |

### How Structured Streaming works (micro-batch)

```
[New files land]  →  [Spark detects them]  →  [Processes as a mini-batch]  →  [Writes output]
      │                     │                          │                            │
    Source              Trigger fires           DataFrame ops              Sink
  (files/Kafka)       (every N seconds)      (filter, join, agg)     (Delta/console)
```

Each micro-batch:
1. Discovers new data since last batch
2. Creates a DataFrame with ONLY the new data
3. Applies your transformations
4. Writes results to the sink
5. Commits the offset to the checkpoint (for exactly-once)

### `readStream` vs `read`

```python
# BATCH: reads everything that exists right now
df_batch = spark.read.json("/path/to/data")

# STREAMING: watches for NEW files continuously
df_stream = spark.readStream.json("/path/to/data")
```

In [0]:
# ============================================================
# STEP 2: Your first streaming query — readStream → writeStream
# ============================================================

# --- Read the stream (watches for new JSON files) ---
df_stream = spark.readStream \
    .schema(event_schema) \
    .json(STREAM_SOURCE)

print("Stream DataFrame schema:")
df_stream.printSchema()
print(f"Is streaming: {df_stream.isStreaming}")

# --- Write the stream to Delta (append mode) ---
# availableNow=True means: process all available data, then stop
# (great for testing — acts like a batch job triggered once)
query = df_stream \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{STREAM_CHECKPOINT}/basic") \
    .trigger(availableNow=True) \
    .start(f"{STREAM_OUTPUT}/events_delta")

# Wait for it to finish
query.awaitTermination()

print(f"\n✅ Stream processed!")
print(f"   Status: {query.status}")
print(f"   Recent progress:")
if query.recentProgress:
    last = query.recentProgress[-1]
    print(f"   - Rows processed: {last.get('numInputRows', 'N/A')}")
    print(f"   - Processing time: {last.get('durationMs', {}).get('triggerExecution', 'N/A')}ms")

# Verify output
df_output = spark.read.format("delta").load(f"{STREAM_OUTPUT}/events_delta")
print(f"\n   Output rows: {df_output.count()}")
df_output.show(5)

### Deep Dive: How Structured Streaming Tracks Progress

**The checkpoint directory** is the brain of your stream. It stores:

```
checkpoint/
├── offsets/     ← Which data has been scheduled for processing
├── commits/     ← Which batches completed successfully
├── metadata    ← Stream ID and query metadata
└── sources/     ← Source-specific state (e.g., which files were seen)
```

**Exactly-once guarantee:**
1. Batch 5 reads new files → writes offset to `offsets/5`
2. Processes the data → writes to sink
3. Records success in `commits/5`
4. If crash happens between steps 2 and 3 → Spark re-does batch 5 (idempotent write)

**NEVER delete the checkpoint directory** of a running stream! You’ll lose track of what’s been processed.

**`availableNow=True` vs continuous running:**
* `availableNow=True` — Process all pending data, then stop. Great for testing or scheduled incremental runs.
* `processingTime="10 seconds"` — Fire a micro-batch every 10 seconds. Stream runs forever.
* `once=True` (deprecated) — Old version of `availableNow`. Don’t use.

**`isStreaming` property:** Returns `True` for streaming DataFrames. Use it to confirm your source is streaming.

## 6.2 — Triggers: When Does Each Batch Fire?

### Trigger types

| Trigger | Behavior | Use case |
| --- | --- | --- |
| `availableNow=True` | Process all available data, then stop | Scheduled incremental ETL (run every hour via a Job) |
| `processingTime="10 seconds"` | Fire every 10s (or as soon as previous batch finishes) | Near-real-time dashboards |
| `processingTime="0 seconds"` | Fire as fast as possible (no waiting) | Lowest latency with micro-batch |
| `continuous="1 second"` | True continuous processing (~1ms latency) | Ultra-low latency (experimental, limited ops) |
| Default (no trigger) | Same as `processingTime="0 seconds"` | Development/testing |

### `availableNow` vs `processingTime`

**`availableNow=True`** (recommended for production ETL):
* Processes ALL pending data (may split into multiple micro-batches internally)
* Stops when caught up — no idle compute sitting around
* Schedule it with a Databricks Job (e.g., every 15 minutes)
* You pay only for active processing time

**`processingTime="10 seconds"`** (for real-time):
* Cluster stays running continuously
* Fires a batch every 10 seconds (even if no new data → empty batch)
* Higher cost (always-on cluster) but lower latency

### When to use what:

```
Latency requirement:
  Minutes OK    → availableNow + scheduled Job (cheapest)
  Seconds       → processingTime="10 seconds" (always-on cluster)
  Milliseconds  → continuous trigger (experimental, limited)
```

In [0]:
# ============================================================
# STEP 3: Triggers — processing new data incrementally
# ============================================================
import time

# --- Add MORE data (batch 2) to simulate new files arriving ---
n = generate_events(2, 150)
print(f"Generated batch 2: {n} new events")

# --- Re-run the stream with availableNow — only processes NEW data ---
print("\nRe-running stream (only new data since last checkpoint):")
df_stream = spark.readStream \
    .schema(event_schema) \
    .json(STREAM_SOURCE)

query = df_stream \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{STREAM_CHECKPOINT}/basic") \
    .trigger(availableNow=True) \
    .start(f"{STREAM_OUTPUT}/events_delta")

query.awaitTermination()

# Check: should now have batch1 + batch2 rows
df_output = spark.read.format("delta").load(f"{STREAM_OUTPUT}/events_delta")
print(f"\n✅ After 2 runs:")
print(f"   Total rows in output: {df_output.count()} (batch1=200 + batch2=150 = 350)")

if query.recentProgress:
    last = query.recentProgress[-1]
    print(f"   Last batch processed: {last.get('numInputRows', 'N/A')} rows (only the NEW ones!)")

print("\n→ Key insight: The checkpoint tracked what was already processed.")
print("  Only batch 2 files were read. Batch 1 was skipped (already done).")

## 6.3 — Output Modes: Which Rows Get Written?

### The 3 output modes

| Mode | What gets written each batch | Use with |
| --- | --- | --- |
| **append** | Only NEW rows from this batch | Filters, maps, simple transforms (no aggregations) |
| **complete** | The ENTIRE result table (recomputed) | Aggregations where you need the full picture |
| **update** | Only rows that CHANGED in this batch | Aggregations where you only want deltas |

### Visual example (counting events by type):

```
Batch 1 arrives: 5 page_views, 3 clicks

  APPEND mode:    ❌ NOT ALLOWED (aggregations produce updated rows, not new rows)
  COMPLETE mode:  writes: {page_view: 5, click: 3}
  UPDATE mode:    writes: {page_view: 5, click: 3}

Batch 2 arrives: 2 page_views, 4 purchases

  COMPLETE mode:  writes: {page_view: 7, click: 3, purchase: 4}  ← entire table
  UPDATE mode:    writes: {page_view: 7, purchase: 4}             ← only changed rows
```

### Which mode to use:

| Scenario | Mode | Why |
| --- | --- | --- |
| Raw event ingestion (no agg) | `append` | Each event is a new row |
| Running totals dashboard | `complete` | Need full current state |
| Incremental updates to Delta | `update` | Only write what changed (efficient) |
| Streaming to console (debug) | `complete` | See full result each time |

In [0]:
# ============================================================
# STEP 4: Output modes — append vs complete vs update
# ============================================================

# --- Generate batch 3 for this demo ---
generate_events(3, 100)
print("Generated batch 3\n")

# Clean checkpoint for this demo
dbutils.fs.rm(f"{STREAM_CHECKPOINT}/output_modes", recurse=True)
dbutils.fs.rm(f"{STREAM_OUTPUT}/complete_mode", recurse=True)
dbutils.fs.rm(f"{STREAM_OUTPUT}/update_mode", recurse=True)

# --- COMPLETE mode: aggregation outputs full result each batch ---
print("COMPLETE MODE (full aggregation result each batch):")
df_stream = spark.readStream.schema(event_schema).json(STREAM_SOURCE)

query_complete = df_stream \
    .groupBy("event_type") \
    .agg(count("*").alias("event_count"), spark_sum("amount").alias("total_amount")) \
    .writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option("checkpointLocation", f"{STREAM_CHECKPOINT}/output_modes/complete") \
    .trigger(availableNow=True) \
    .start(f"{STREAM_OUTPUT}/complete_mode")

query_complete.awaitTermination()

df_complete = spark.read.format("delta").load(f"{STREAM_OUTPUT}/complete_mode")
print("Result (entire aggregation table):")
df_complete.orderBy("event_type").show()

# --- UPDATE mode: use memory sink (Delta doesn't support update mode) ---
print("UPDATE MODE (using memory sink — Delta only supports append/complete):")
df_stream2 = spark.readStream.schema(event_schema).json(STREAM_SOURCE)

query_update = df_stream2 \
    .groupBy("event_type") \
    .agg(count("*").alias("event_count"), spark_sum("amount").alias("total_amount")) \
    .writeStream \
    .format("memory") \
    .queryName("update_mode_demo") \
    .outputMode("update") \
    .option("checkpointLocation", f"{STREAM_CHECKPOINT}/output_modes/update") \
    .trigger(availableNow=True) \
    .start()

query_update.awaitTermination()

print("Result from memory sink (update mode):")
spark.sql("SELECT * FROM update_mode_demo ORDER BY event_type").show()

print("→ Same numbers as complete mode (first batch processes all data).")
print("  The difference shows on SUBSEQUENT batches:")
print("  - COMPLETE: rewrites the FULL table each time")
print("  - UPDATE: only outputs rows whose counts CHANGED")
print("\n  Note: Delta sink supports append + complete. For update mode,")
print("  use memory/console sinks (testing) or foreachBatch (production).")

### Deep Dive: Output Mode Rules & Restrictions

**What operations are allowed in each mode:**

| Operation | append | complete | update |
| --- | --- | --- | --- |
| `select`, `filter`, `map` | ✅ | ✅ | ✅ |
| `groupBy().agg()` | ❌ | ✅ | ✅ |
| `dropDuplicates` | ✅ | ❌ | ✅ |
| Windowed aggregation + watermark | ✅ (after watermark) | ✅ | ✅ |
| `flatMap` / `explode` | ✅ | ❌ | ❌ |

**Why can’t aggregations use append mode?**

Append means “only output NEW rows that will never change.” But `groupBy().count()` updates existing rows each batch (the count increases). Spark can’t guarantee those rows won’t change, so it rejects append.

**Exception:** Windowed aggregations WITH a watermark CAN use append. Once the watermark passes a window, that window is “final” and can be safely appended.

**Complete mode + Delta = overwrite:**
When using complete mode with Delta sink, each batch REPLACES the entire table. This is fine for small aggregation results but wasteful for large outputs.

**Production recommendation:**
* Raw ingestion: `append` to Delta
* Aggregations feeding dashboards: `complete` (small result sets) or `update` (large)
* Incremental ETL: `append` with `availableNow=True`

## 6.4 — Watermarks: Handling Late Data

### The problem

Events arrive late. A user’s click at 10:00:05 might not reach your system until 10:02:00 due to:
* Network delays
* Mobile app buffering
* System backlogs

If you’re computing `window("event_time", "5 minutes")`, should you keep the 10:00–10:05 window open forever waiting for late data?

### Watermarks = “how late is too late?”

```python
df_stream.withWatermark("event_time", "10 minutes")
```

This says: “I’ll wait up to 10 minutes for late data. After that, drop it.”

**How Spark uses the watermark:**
1. Tracks the MAX event_time seen so far (say 10:30:00)
2. Watermark = max_event_time - threshold = 10:30:00 - 10min = 10:20:00
3. Any event with event_time < 10:20:00 is “too late” → dropped
4. Windows older than watermark are finalized and can be output in append mode

### Why watermarks matter:

| Without watermark | With watermark |
| --- | --- |
| Spark keeps ALL windows open forever | Old windows get closed and cleaned up |
| State grows unbounded → OOM | State is bounded → stable memory |
| Can’t use append mode for windowed aggs | CAN use append mode (windows are “final”) |
| No late data handling | Late data handled gracefully |

In [0]:
# ============================================================
# STEP 5: Watermarks + windowed aggregation
# ============================================================
from pyspark.sql.functions import window, count, sum as spark_sum

# Clean up
dbutils.fs.rm(f"{STREAM_CHECKPOINT}/watermark", recurse=True)
dbutils.fs.rm(f"{STREAM_OUTPUT}/windowed_counts", recurse=True)

# --- Generate fresh data with known timestamps ---
generate_events(4, 300)

# --- Streaming with watermark + window ---
print("WINDOWED AGGREGATION with watermark:")
print("  Window: 1 minute tumbling")
print("  Watermark: 2 minutes (accept data up to 2 min late)\n")

df_stream = spark.readStream \
    .schema(event_schema) \
    .json(STREAM_SOURCE)

# Apply watermark on event_time, then window
df_windowed = df_stream \
    .withWatermark("event_time", "2 minutes") \
    .groupBy(
        window(col("event_time"), "1 minute"),  # 1-minute tumbling window
        col("event_type")
    ) \
    .agg(
        count("*").alias("event_count"),
        spark_sum("amount").alias("total_amount")
    )

# Use append mode (possible because watermark makes windows "final")
query = df_windowed \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{STREAM_CHECKPOINT}/watermark") \
    .trigger(availableNow=True) \
    .start(f"{STREAM_OUTPUT}/windowed_counts")

query.awaitTermination()

# Show results
df_result = spark.read.format("delta").load(f"{STREAM_OUTPUT}/windowed_counts")
print(f"Windowed aggregation output: {df_result.count()} window-event_type combinations")
df_result.select(
    col("window.start").alias("window_start"),
    col("window.end").alias("window_end"),
    "event_type", "event_count", "total_amount"
).orderBy("window_start", "event_type").show(10, truncate=False)

print("→ Each row = count of events per type within a 1-minute window.")
print("  The watermark ensures windows close after 2 minutes of no late data.")
print("  Closed windows are output in APPEND mode (they won't change again).")

### Deep Dive: Watermark Mechanics

**Step-by-step example:**

Watermark threshold: 10 minutes. Window: 5 minutes.

```
Batch 1: max event_time = 10:30:00
  Watermark = 10:30 - 10min = 10:20:00
  Windows still open: [10:20-10:25], [10:25-10:30]
  Windows closed: everything before 10:20

Batch 2: max event_time = 10:40:00
  Watermark = 10:40 - 10min = 10:30:00
  A late event arrives with event_time = 10:18:00
  → 10:18 < watermark (10:30) → DROPPED! Too late.

  Another late event: event_time = 10:32:00
  → 10:32 > watermark (10:30) → ACCEPTED (within tolerance)
```

**Choosing the right watermark threshold:**

| Threshold | Trade-off |
| --- | --- |
| Too short (1 min) | Low memory, fast window closure, but drops many late events |
| Too long (1 hour) | Catches late events, but holds lots of state in memory |
| Just right | Depends on your data’s lateness distribution |

**How to decide:** Look at your actual data lateness:
```python
# In batch mode, measure how late events typically are:
df.select(
    (current_timestamp() - col("event_time")).alias("lateness")
).describe().show()
```
Set watermark slightly above your 99th percentile lateness.

**Memory impact:** Without watermarks, state for ALL windows is kept forever. With a 10-minute watermark + 5-minute windows, Spark only keeps ~15 minutes of state. Critical for long-running streams!

## 6.5 — Stateful Operations: Windows & Sessions

### Types of windowed operations:

| Window Type | What it does | Example |
| --- | --- | --- |
| **Tumbling** | Fixed-size, non-overlapping | `window("ts", "5 minutes")` — [0:00-0:05], [0:05-0:10] |
| **Sliding** | Fixed-size, overlapping | `window("ts", "10 min", "5 min")` — [0:00-0:10], [0:05-0:15] |
| **Session** | Variable-size, gap-based | Events within 30min of each other grouped together |

### Tumbling vs Sliding:

```
Tumbling (5 min):
|--0:00--0:05--|--0:05--0:10--|--0:10--0:15--|
Each event belongs to exactly ONE window.

Sliding (10 min window, 5 min slide):
|--0:00----------0:10--|
     |--0:05----------0:15--|
          |--0:10----------0:20--|
Each event can belong to MULTIPLE windows.
```

### Session windows (gap-based):
```python
from pyspark.sql.functions import session_window

df_stream.withWatermark("event_time", "10 minutes") \
    .groupBy(session_window("event_time", "30 minutes"), "user_id") \
    .agg(count("*"))
```
Groups events by user where gaps between events are < 30 minutes. New gap = new session.

In [0]:
# ============================================================
# STEP 6: Sliding window — overlapping time windows
# ============================================================
from pyspark.sql.functions import window, count, col

# Clean up
dbutils.fs.rm(f"{STREAM_CHECKPOINT}/sliding", recurse=True)
dbutils.fs.rm(f"{STREAM_OUTPUT}/sliding_windows", recurse=True)

# Generate more data
generate_events(5, 200)

# --- Sliding window: 2-minute window, sliding every 1 minute ---
print("SLIDING WINDOW: 2-minute window, sliding every 1 minute")
print("  Each event appears in up to 2 windows (overlapping)\n")

df_stream = spark.readStream \
    .schema(event_schema) \
    .json(STREAM_SOURCE)

df_sliding = df_stream \
    .withWatermark("event_time", "3 minutes") \
    .groupBy(
        window(col("event_time"), "2 minutes", "1 minute"),  # 2min window, 1min slide
        col("event_type")
    ) \
    .agg(count("*").alias("event_count"))

query = df_sliding \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{STREAM_CHECKPOINT}/sliding") \
    .trigger(availableNow=True) \
    .start(f"{STREAM_OUTPUT}/sliding_windows")

query.awaitTermination()

df_result = spark.read.format("delta").load(f"{STREAM_OUTPUT}/sliding_windows")
print(f"Sliding window results: {df_result.count()} rows")
df_result.select(
    col("window.start").alias("window_start"),
    col("window.end").alias("window_end"),
    "event_type", "event_count"
).orderBy("window_start", "event_type").show(10, truncate=False)

print("→ Notice: same event_type appears in adjacent windows (overlapping!).")
print("  Sliding windows let you compute 'events in the last 2 min' every minute.")

## 6.6 — Stream-to-Stream Joins

### When you need to join two streams:

**Example:** Match `clicks` stream with `impressions` stream to compute click-through rate.

Both streams have events arriving at different times. Spark must buffer state from both sides until a match is found.

### Rules for stream-stream joins:

| Join Type | Watermark Required? | Notes |
| --- | --- | --- |
| Inner join | Optional (but recommended) | Without watermark, state grows forever |
| Left outer | Required on RIGHT side | Need to know when right side won’t get more matches |
| Right outer | Required on LEFT side | Need to know when left side won’t get more matches |
| Full outer | Required on BOTH sides | Both sides need bounded wait |

### Example:
```python
# Both streams need watermarks for outer joins
df_clicks = clicks_stream.withWatermark("click_time", "10 minutes")
df_impressions = impressions_stream.withWatermark("imp_time", "10 minutes")

# Join with time constraint
df_joined = df_clicks.join(
    df_impressions,
    expr("""
        click_user_id = imp_user_id AND
        click_time >= imp_time AND
        click_time <= imp_time + interval 5 minutes
    """),
    how="inner"
)
```

The time constraint (`click within 5 min of impression`) limits how long Spark buffers state — old impressions with no click after 5 minutes are dropped.

In [0]:
# ============================================================
# STEP 7: Stream-to-stream join (clicks × impressions)
# ============================================================
from pyspark.sql.functions import expr

# Clean up
dbutils.fs.rm(f"{STREAM_CHECKPOINT}/stream_join", recurse=True)
dbutils.fs.rm(f"{STREAM_OUTPUT}/stream_join_result", recurse=True)
dbutils.fs.rm(f"{VOLUME_PATH}/impressions_stream", recurse=True)
dbutils.fs.rm(f"{VOLUME_PATH}/clicks_stream", recurse=True)

# --- Generate two correlated streams ---
# Impressions: shown to users
df_impressions = spark.range(500).select(
    expr("concat('imp_', id)").alias("impression_id"),
    (floor(rand() * 30) + 1).cast("int").alias("user_id"),
    expr("""CASE floor(rand() * 3)
        WHEN 0 THEN 'banner_ad'
        WHEN 1 THEN 'sidebar_ad'
        ELSE 'popup_ad'
    END""").alias("ad_type"),
    expr("current_timestamp() - make_interval(0,0,0,0,0, floor(rand() * 300), 0)").alias("imp_time")
)
df_impressions.write.mode("overwrite").json(f"{VOLUME_PATH}/impressions_stream")

# Clicks: ~20% of impressions get clicked (with slight delay)
df_clicks = spark.range(100).select(
    expr("concat('click_', id)").alias("click_id"),
    expr("concat('imp_', floor(rand() * 500))").alias("impression_id"),
    (floor(rand() * 30) + 1).cast("int").alias("user_id"),
    # Clicks happen 10-120 seconds after impression
    expr("current_timestamp() - make_interval(0,0,0,0,0, floor(rand() * 200), 0)").alias("click_time")
)
df_clicks.write.mode("overwrite").json(f"{VOLUME_PATH}/clicks_stream")

print(f"Generated: 500 impressions, 100 clicks")

# --- Define schemas ---
imp_schema = "impression_id STRING, user_id INT, ad_type STRING, imp_time TIMESTAMP"
click_schema = "click_id STRING, impression_id STRING, user_id INT, click_time TIMESTAMP"

# --- Read both as streams ---
df_imp_stream = spark.readStream.schema(imp_schema).json(f"{VOLUME_PATH}/impressions_stream")
df_click_stream = spark.readStream.schema(click_schema).json(f"{VOLUME_PATH}/clicks_stream")

# --- Apply watermarks and rename to avoid ambiguity ---
df_imp_wm = df_imp_stream \
    .withColumnRenamed("impression_id", "imp_id") \
    .withColumnRenamed("user_id", "imp_user_id") \
    .withWatermark("imp_time", "5 minutes")

df_click_wm = df_click_stream \
    .withColumnRenamed("user_id", "click_user_id") \
    .withWatermark("click_time", "5 minutes")

# --- Join: match clicks to impressions within 5-minute window ---
df_joined = df_imp_wm.join(
    df_click_wm,
    (df_imp_wm["imp_id"] == df_click_wm["impression_id"]) &
    (df_click_wm["click_time"] >= df_imp_wm["imp_time"]) &
    (df_click_wm["click_time"] <= df_imp_wm["imp_time"] + expr("interval 5 minutes")),
    how="inner"
)

# Write joined result
query = df_joined \
    .select("imp_id", "click_id", "ad_type", "imp_time", "click_time") \
    .writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{STREAM_CHECKPOINT}/stream_join") \
    .trigger(availableNow=True) \
    .start(f"{STREAM_OUTPUT}/stream_join_result")

query.awaitTermination()

# Show results
df_result = spark.read.format("delta").load(f"{STREAM_OUTPUT}/stream_join_result")
print(f"\n✅ Stream-to-stream join result: {df_result.count()} matched click-impression pairs")
df_result.show(5, truncate=False)
print("→ Only clicks within 5 minutes of their impression were matched.")
print("  Watermarks ensure Spark doesn't buffer state forever.")

### Deep Dive: State Management in Streaming

**Every stateful operation keeps data in memory:**

| Operation | State stored | Growth |
| --- | --- | --- |
| `groupBy().agg()` | Running aggregates per key | Bounded by distinct keys |
| Windowed aggregation | Aggregates per window + key | Bounded by watermark |
| Stream-stream join | Buffered rows from both sides | Bounded by watermark/time constraint |
| `dropDuplicates` | Seen keys | Bounded by watermark |

**Without watermarks:** State grows forever → eventually OOM.

**State store backends:**
* Default: in-memory (HDFS-backed for fault tolerance)
* RocksDB: disk-based state store (handles larger state)

```python
# Use RocksDB for large state (configure in Spark conf)
spark.conf.set("spark.sql.streaming.stateStore.providerClass",
    "com.databricks.sql.streaming.state.RocksDBStateStoreProvider")
```

**Monitoring state size (Spark UI → Structured Streaming tab):**
* `numRowsTotal` — total keys in state
* `stateMemory` — memory used by state
* If growing over time without watermark → you have a state leak

**Production checklist for streaming:**
1. ✅ Always set a watermark on event-time columns
2. ✅ Use `availableNow=True` for scheduled incremental ETL
3. ✅ Never delete checkpoint directories of active streams
4. ✅ Monitor state size in Spark UI
5. ✅ Use RocksDB state store for large state (>1GB)
6. ✅ Set appropriate trigger interval (don’t fire faster than you can process)

## Phase 6 Checkpoint

**Can you do these?**

- [ ] Explain the difference between `spark.read` and `spark.readStream`
- [ ] Set up a streaming query with `readStream` → transform → `writeStream`
- [ ] Choose the right trigger (`availableNow`, `processingTime`, or default)
- [ ] Explain all 3 output modes (append, complete, update) and when to use each
- [ ] Use `withWatermark()` to handle late-arriving data
- [ ] Create tumbling and sliding window aggregations
- [ ] Explain why checkpoints enable exactly-once processing
- [ ] Join two streams with time constraints and watermarks
- [ ] Explain state management and why watermarks prevent OOM

---

**Key takeaways:**
1. **Same API, different execution** — `readStream`/`writeStream` uses the same DataFrame ops you know
2. **`availableNow=True`** — best trigger for scheduled ETL (process & stop, no idle compute)
3. **Checkpoints = exactly-once** — never delete them, they track what’s been processed
4. **Watermarks = bounded state** — without them, memory grows forever
5. **Output modes matter** — `append` for raw data, `complete`/`update` for aggregations

---

**Next up:** Phase 7 — Advanced Transformations (complex types, higher-order functions, pivot/unpivot, nested data)

---